# L2 step 2b · Why the 3B's training gets NaN in half precision, and which fix works (GPU T4 ×2, about 45 minutes)

Task **L2** (distilling our fine-tuned 7B into the smaller 3B), a diagnostic for its step 2, the 3B's training (`l2-step2-train-3b.ipynb`). Built 19 Nov 2026 at Abrar's go. Nothing it trains is kept.

**The question:** on which examples, and in which layer, does the 3B's half-precision forward pass overflow, and which fix stops it without changing the loss?

**Why**
- Step 2's run 3 stopped after 16 optimizer steps (4 h of training) with nothing learned. That was option (c): the 3B trained on L2 step 1's checked answers plus the options rerun's 806 examples, on account 1, 18 Nov. At every step, `grad_norm` was NaN and the learning rate stayed 0. The trainer moves the learning rate only after a real update, so every update was skipped.
- `label_loss` is the mean over the first GPU's 128 examples in a step. It was NaN in 13 of the 16 steps; the total loss the trainer logged stayed finite only because it filters NaN out.
- Every example has answer tokens, so some examples' forward pass overflows in half precision (fp16: largest value 65,504). Qwen2.5-VL-3B is known to do this in fp16.
- 13 of 16 steps with a NaN among 128 examples means about **1.3%** of examples. A global batch of 256 then holds about 3, and one is enough to skip the update.
- The run also spent 33 minutes on its data step before training (its log: 105 s → 2,106 s). Most of that is checking image files one at a time on Kaggle's input disk.

**What it settles:** which fix step 2's restart uses, what that fix costs in speed, and how much of a session is left for training.

**What it does**
1. **The same examples, batches and loss as the run.**
   - Its data rules, its batches (`collate`) and its loss (`losses`) are copied from step 2's Cell 3 when this notebook is built.
   - The 3B is loaded as there: `Qwen/Qwen2.5-VL-3B-Instruct`, 4-bit NF4, fp16 compute, fp16 weights elsewhere, SDPA attention, run under fp16 autocast as the trainer runs it.
   - The adapter is left out of the scan: at the start it adds exactly zero, and the failed run never left the start.
2. **Faster image finding, same answers.** The same rule, but with each folder listed once instead of each file checked. Step 2's own asserts still hold (55,764 eligible entries, 53,032 for training), and the images of 300 spread examples are checked on disk.
3. **The scan, as trained:** 50 seeded examples per task (8 tasks) plus the 40 longest (most images, longest answer), 440 in all. At 1.3% that is about 6 NaN.
   - Every module's output is watched: its largest absolute value, and whether it is finite.
   - Per example: the label loss on the answer tokens, and where the overflow starts. That is the first module, in running order, whose output isn't finite.
   - If that module's input was already not finite, an operation in its parent's own code made it (an attention score, a residual sum), so the parent is named.
4. **Three fixes,** tested on every example the scan found overflowing plus 15 others per task (seeded):
   - **(a) fp32 islands:** the modules where the overflow starts run in full precision (fp32), with autocast off. If another module then overflows, it joins them (up to 3 rounds). After them, the residual stream stays in fp32.
   - **(b) clamping:** outputs are clamped to ±(65,504 − 1,000), as T5 does in fp16. This applies after every decoder layer, after its attention and its MLP, and after every vision block, so an infinity never reaches the next normalisation.
   - **(c) fp32 compute (the reference):** the model loaded with fp32 compute and fp32 weights on the second T4, without autocast.

   For each fix: the examples with a NaN loss, the seconds per example, and the mean distance of its loss from (c)'s.
5. **Learning:** with the chosen fix, 8 optimizer steps of 4 examples, the overflowing ones first. The trainer's fp16 mechanics are kept:
   - a gradient scaler starting at 2^16;
   - AdamW 8-bit, lr 2e-4 with 1 warm-up step, then cosine;
   - clipping at 1.0;
   - the learning rate stepped only after a real update.

**The rule (fixed before the run)**
- **The fix:** of (a) and (b), those with no NaN loss on the test examples and a mean loss distance from (c) of at most 0.02; of those, the faster. If neither qualifies, (c).
- **Learning works** if, within the 8 steps, at least one update is real (learning rate above 0), and the last 3 steps have a finite `grad_norm`.
- **If the scan finds no NaN,** it says so: the run's NaN examples are rarer than 440 can show. The fixes are still compared the same way, as a guard.

**Expectation (fixed before the run)**
- **As trained:** a NaN loss on about 1–2% of examples, weighted by task share.
- **Where it starts:** in an MLP's output (`down_proj`, or the product before it). Most likely in the language model's late layers, the "massive activations" these models are known for; otherwise in the vision tower.
- **Fixes:** (a) and (b) both remove every NaN, within 0.01 of (c)'s loss. (b) costs under 5% in speed; (c) is 2–4 times slower.
- **Learning:** with the fix, the learning rate leaves 0 within the first 5 steps, and `grad_norm` turns finite.
- **Data:** the image finding takes a minute or two, not 33.

**How to run on Kaggle** (account 1: step 2's run 3 had these inputs attached)
1. Settings: GPU **T4 x2** (the reference uses the second one), Internet **On**. The `HF_TOKEN` secret is optional, since the 3B's repository is public.
2. Inputs, the same three as step 2's run 3:
   - `abrarmohammedtanzim/disasterm3-mirror`;
   - the output of `l2-step1-new-teacher`;
   - the output of `l2-step1c-options-rerun`.
3. Run All once with `SMOKE_TEST = True` in Cell 2: 2 examples per task and 2 learning steps, about 10 minutes. Every cell should print its ✓ line.
4. Then set `SMOKE_TEST = False` and use **Save Version → Save & Run All (Commit)**. About 45 minutes, estimated; nearer an hour if (c) is the fix, since its learning steps then run in fp32.

**Outputs** in `/kaggle/working/L2S2b/`:
- `README.md`, `config.json`;
- `per_example.csv`: each example's task, images, tokens, its loss as trained and under each fix, and where its overflow starts;
- `modules.csv`: per module, how often the overflow started there, and its largest finite output;
- `fixes.csv`: per fix, the NaN losses, the seconds per example and the distance from (c);
- `train_steps.csv`: the 8 learning steps (loss, `grad_norm`, learning rate, scale, skipped or not).

In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, plus current peft and bitsandbytes (as step 2) ──
import importlib.metadata as _md
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")      # as step 2's train.py
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes"], check=True)


def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"


assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16 and later)"

import torch

VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the fp32 reference runs on the second GPU"

✓ {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'peft': '0.21.3', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0'}, python 3.13.16
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration (step 2's, option (c)), the inputs, and the base model in the cache ──
import glob
import json
from pathlib import Path

from huggingface_hub import snapshot_download

SMOKE_TEST = False             # True = 2 examples per task and the 2 longest, 2 learning steps of 2 examples
MODE = "c"                     # the run that failed: option (c)
WORK = Path("/kaggle/working/L2S2b")
WORK.mkdir(parents=True, exist_ok=True)


def one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return hits[0]


CFG = {
    "task": "L2 step 2b: fp16 overflow diagnostic for step 2, option (c)", "mode": MODE, "smoke_test": SMOKE_TEST,
    "base_model": "Qwen/Qwen2.5-VL-3B-Instruct",
    "data_root": "/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct",
    "l2s1_manifest": one("L2S1-new-teacher/disasterm3_distillation_cot_train.json", "attach the output of l2-step1-new-teacher"),
    "options_kept": one("L2S1c/options_kept.jsonl", "attach the output of l2-step1c-options-rerun"),
    "seed": 42,
    "lora": {"r": 64, "alpha": 16, "dropout": 0.05},                 # step 2's
    "lr": 2e-4, "betas": [0.9, 0.95], "weight_decay": 0.01,           # step 2's
    "kd_alpha": 0.5, "kd_topk": 20, "smoke_examples": 16,             # read by step 2's code; no effect in option (c)
    "quantize": True, "fp16": True,
    "per_task": 2 if SMOKE_TEST else 50, "longest": 2 if SMOKE_TEST else 40, "test_per_task": 1 if SMOKE_TEST else 15,
    "learn_steps": 2 if SMOKE_TEST else 8, "learn_accum": 2 if SMOKE_TEST else 4,
    "init_scale": 2.0 ** 16, "max_grad_norm": 1.0,                    # the trainer's
    "clamp_margin": 1000.0, "max_loss_gap": 0.02,
}
(WORK / "config.json").write_text(json.dumps(CFG, indent=1), encoding="utf-8")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                       # the 3B's repository is public
BASE = snapshot_download(CFG["base_model"], token=os.environ.get("HF_TOKEN"))
print(f"✓ L2 step 2b{'  [SMOKE TEST]' if SMOKE_TEST else ''}: base cached at {BASE}")
print(f"  Inputs: {CFG['l2s1_manifest']}, {CFG['options_kept']}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

✓ L2 step 2b: base cached at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3
  Inputs: /kaggle/input/notebooks/abrarmohammedtanzim/l2-step1-new-teacher/L2S1-new-teacher/disasterm3_distillation_cot_train.json, /kaggle/input/notebooks/abrar2222864642/l2-step1c-options-rerun/L2S1c/options_kept.jsonl


In [3]:
# ── Cell 3: Step 2's data, batches and loss (copied from its Cell 3 when this notebook was built), and the sample ──
import hashlib
import math
import random
import re
import time
from collections import Counter
from functools import lru_cache

import numpy as np
from PIL import Image
from transformers import AutoProcessor

SMOKE, RANK, WORLD, MAIN = False, 0, 1, True      # step 2's names: one process, and its own smoke sampling off
DATA_ROOT = Path(CFG["data_root"])
TRAIN_ENTRIES = 53032


def log(msg):
    print(msg, flush=True)


# Step 2's data step took 33 min (its log: 105 s -> 2,106 s), checking image files one by one on Kaggle's input disk.
# Here each folder is listed once and the same rule reads the listing; step 2's own counts are asserted below.
@lru_cache(maxsize=None)
def _names_in(folder):
    try:
        return frozenset(os.listdir(folder))
    except OSError:
        return frozenset()


_real_exists = Path.exists
Path.exists = lambda self, *a, **k: self.name in _names_in(str(self.parent))
_t0 = time.time()

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})

# ── Data: the same 53,032 training entries for both options; each option's answers and extras ──
raw = json.load(open(DATA_ROOT / "train_release.json", encoding="utf-8"))
eligible_entries = [e for e in raw if eligible(e)]
del raw
assert len(eligible_entries) == ELIGIBLE, f"{len(eligible_entries):,} eligible entries, not {ELIGIBLE:,}"
train_idx = [i for i, e in enumerate(eligible_entries) if not held_out(e)]
assert len(train_idx) == TRAIN_ENTRIES, f"{len(train_idx):,} training entries, not {TRAIN_ENTRIES:,}"

def prompt_of(e):
    p = e.get("prompts", "")
    return str(p[0] if isinstance(p, list) else p)

def images_of(e):
    keys = ["image_path"] if "image_path" in e else ["pre_image_path", "post_image_path"]
    return [resolve_image_path(e[k]) for k in keys]

SOFT, teacher_answers = {}, 0
if MODE == "b":
    answers = {i: answer_text(eligible_entries[i]) for i in train_idx}
    for p in sorted(Path(CFG["soft_labels_dir"]).glob("*.npz")):
        with np.load(p) as z:
            ent, off = z["entry_idx"], z["offsets"]
            tgt, ids, lp = z["target_ids"], z["topk_ids"], z["topk_logp"]
            for j, i in enumerate(ent.tolist()):
                s, t = off[j], off[j + 1]
                SOFT[i] = (tgt[s:t].astype(np.int64), ids[s:t, :CFG["kd_topk"]].astype(np.int64), lp[s:t, :CFG["kd_topk"]])
    leaked = [i for i in SOFT if held_out(eligible_entries[i])]
    assert not leaked, f"{len(leaked)} soft labels belong to held-out entries"
    assert SMOKE or len(SOFT) == CHECKABLE_TRAIN, f"{len(SOFT):,} soft labels, not {CHECKABLE_TRAIN:,}: is step 1b complete?"
    extras = []
else:
    l2s1 = json.load(open(CFG["l2s1_manifest"], encoding="utf-8"))
    assert len(l2s1) == ELIGIBLE and all(l2s1[i]["prompts"] == eligible_entries[i]["prompts"] for i in train_idx[::97]), \
        "L2 step 1's manifest isn't in the eligible entries' order"
    answers = {i: (l2s1[i].get("teacher_rationale") or answer_text(eligible_entries[i])) for i in train_idx}
    teacher_answers = sum(1 for i in train_idx if l2s1[i].get("teacher_rationale"))    # L2 step 1's kept answers
    del l2s1
    with open(CFG["options_kept"], encoding="utf-8") as f:
        extras = [json.loads(line) for line in f if line.strip()]
    assert not [x for x in extras if held_out(eligible_entries[x["entry_idx"]])], "an options example is on a held-out photo"

examples = [{"eidx": i, "images": images_of(eligible_entries[i]), "marked": None, "prompt": prompt_of(eligible_entries[i]),
             "answer": answers[i], "task": eligible_entries[i]["task"], "kind": "train"} for i in train_idx]
examples += [{"eidx": x["entry_idx"], "images": [resolve_image_path(p) for p in x["images"]], "marked": x["prompt"], "prompt": None,
              "answer": x["answer"], "task": x["task"], "kind": "options"} for x in extras]
by_kind = dict(Counter(x["kind"] for x in examples))
by_task = dict(Counter(x["task"] for x in examples))
teacher_reworded = sum(1 for i in train_idx if answers[i] != answer_text(eligible_entries[i]))   # of those, worded unlike the label
longest =max(examples, key=lambda x: (len(x["images"]), len(x["answer"])))
if SMOKE:        # the longest (peak memory), 8 with soft labels or options examples, and others
    rng = random.Random(CFG["seed"])
    special = [x for x in examples if (MODE == "b" and x["eidx"] in SOFT) or x["kind"] == "options"]
    plain = [x for x in examples if x not in special and x is not longest]
    pick = rng.sample(special, min(8, len(special)))
    examples = [longest] + pick + rng.sample(plain, CFG["smoke_examples"] - 1 - len(pick))
log(f"✓ Data, option ({MODE}): {sum(by_kind.values()):,} examples {by_kind}; by task {by_task}"
    + (f"; soft labels for {len(SOFT):,}" if MODE == "b" else
       f"; the new 7B's checked answers in {teacher_answers:,} ({teacher_reworded:,} worded unlike the label)")
    + (f"; SMOKE TEST: {len(examples)} of them" if SMOKE else ""))

Path.exists = _real_exists
DATA_SECONDS = time.time() - _t0
_spread = examples[:: max(1, len(examples) // 300)]
_missing = [p for x in _spread for p in x["images"] if not (p and Path(p).exists())]
assert not _missing, f"{len(_missing)} image paths of the spread examples aren't on disk: {_missing[:3]}"
log(f"✓ Data step in {DATA_SECONDS:.0f} s (step 2's: about 2,000 s); the images of {len(_spread)} spread examples are all on disk")

# ── Batches: the chat template, the answer-only loss, and (b) the 7B's soft labels lined up with the answer tokens ──
processor = AutoProcessor.from_pretrained(CFG["base_model"])
tok = processor.tokenizer
MARKER = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
VISION_IDS = torch.tensor([tok.convert_tokens_to_ids(t) for t in ("<|image_pad|>", "<|vision_start|>", "<|vision_end|>")])
K = CFG["kd_topk"]

def mask_prompt(row):
    """No loss up to and including the last assistant header: the loss is on the answer only, as in the 7B's training."""
    ids = row.tolist()
    for start in range(len(ids) - len(MARKER), -1, -1):
        if ids[start:start + len(MARKER)] == MARKER:
            row[:start + len(MARKER)] = -100
            return row
    raise ValueError("no assistant turn in the sequence")

def content_of(x, n_images):
    if x["marked"]:                     # (c)'s options examples: the benchmark prompt, a picture in each '<image>' gap
        parts, content = x["marked"].split("<image>"), []
        for n, part in enumerate(parts):
            if part:
                content.append({"type": "text", "text": part})
            if n < len(parts) - 1:
                content.append({"type": "image"})
        return content
    return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": x["prompt"]}]

def collate(batch):
    texts, images = [], []
    for x in batch:
        imgs = [sized(Image.open(p).convert("RGB")) for p in x["images"]]
        messages = [{"role": "user", "content": content_of(x, len(imgs))},
                    {"role": "assistant", "content": [{"type": "text", "text": x["answer"]}]}]
        texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
        images += imgs
    out = processor(text=texts, images=images, padding=True, return_tensors="pt")
    labels = out["input_ids"].clone()
    labels[out["attention_mask"] == 0] = -100
    for row in labels:
        mask_prompt(row)
    assert not torch.isin(labels, VISION_IDS).any(), "image tokens in the loss"
    # The output layer (152k tokens) runs only from the token before the first answer token to the end (L0's saving)
    keep = labels.shape[1] - int((labels != -100).int().argmax(1).min()) + 1
    out["labels"], out["logits_to_keep"] = labels[:, -keep:], keep
    if MODE == "b":                     # the 7B's top-k per answer token; -1 where a row has none
        soft = [SOFT.get(x["eidx"]) if x["kind"] == "train" else None for x in batch]
        T = max([len(s[0]) for s in soft if s is not None], default=1)
        kd_target = torch.full((len(batch), T), -1, dtype=torch.long)
        kd_ids = torch.zeros((len(batch), T, K), dtype=torch.long)
        kd_logp = torch.zeros((len(batch), T, K), dtype=torch.float32)
        for b, s in enumerate(soft):
            if s is not None:
                n = len(s[0])
                kd_target[b, :n] = torch.from_numpy(s[0])
                kd_ids[b, :n] = torch.from_numpy(s[1])
                kd_logp[b, :n] = torch.from_numpy(s[2].astype(np.float32))
        out["kd_target"], out["kd_ids"], out["kd_logp"] = kd_target, kd_ids, kd_logp
    return out

def losses(logits, labels, kd=None, alpha=CFG["kd_alpha"]):
    """(loss, label loss, soft loss or None, soft tokens, misaligned tokens). logits[:, t] predicts labels[:, t + 1]."""
    logits, targets = logits[:, :-1].float(), labels[:, 1:]
    valid = targets != -100
    logp = torch.log_softmax(logits[valid], -1)
    tgt = targets[valid]
    ce = -logp.gather(1, tgt[:, None]).mean()
    if not kd:
        return ce, ce, None, 0, 0
    rows = valid.nonzero()[:, 0]
    pos = (valid.long().cumsum(1) - 1)[valid]          # each answer token's place in its row's answer
    kt = kd["kd_target"].to(tgt.device)
    inside = pos < kt.shape[1]
    has = torch.zeros_like(inside)
    has[inside] = kt[rows[inside], pos[inside]] >= 0
    r, p = rows[has], pos[has]
    aligned = kt[r, p] == tgt[has]
    misaligned = int((~aligned).sum())
    use = has.clone()
    use[has] = aligned
    if not use.any():
        return ce, ce, None, 0, misaligned
    r, p = rows[use], pos[use]
    p_t = torch.softmax(kd["kd_logp"].to(logp.device)[r, p], -1)         # the 7B's top-k, renormalised (temperature 1)
    s_lp = logp[use].gather(1, kd["kd_ids"].to(logp.device)[r, p])       # the 3B's log-probabilities of those tokens
    soft = -(p_t * s_lp).sum(-1).mean()
    return (1 - alpha) * ce + alpha * soft, ce, soft, int(use.sum()), misaligned

# Self-checks of the loss on hand-made numbers: a perfect student, the alignment guard, the weighting
_V = 6
_lab = torch.tensor([[-100, -100, 2, 3, 4]])
_lg = torch.full((1, 5, _V), -20.0)
for _t, _c in ((1, 2), (2, 3), (3, 4)):
    _lg[0, _t, _c] = 20.0
_kd = {"kd_target": torch.tensor([[2, 3, 4]]), "kd_ids": torch.tensor([[[2, 0], [3, 0], [4, 0]]]),
       "kd_logp": torch.tensor([[[0.0, -50.0]] * 3])}
_l, _ce, _soft, _n, _bad = losses(_lg, _lab, _kd)
assert _ce < 1e-6 and _soft < 1e-6 and _n == 3 and _bad == 0
_kd_bad = dict(_kd, kd_target=torch.tensor([[2, 5, 4]]))
assert losses(_lg, _lab, _kd_bad)[3:] == (2, 1)                 # the mismatched token gets no soft loss and is counted
_lg2 = torch.zeros((1, 5, _V))
_l2, _ce2, _soft2, _, _ = losses(_lg2, _lab, _kd)
assert abs(_l2 - (0.5 * _ce2 + 0.5 * _soft2)) < 1e-6 and abs(_ce2 - math.log(_V)) < 1e-5

probe = collate([longest])
log(f"✓ Longest example: {probe['input_ids'].shape[1]:,} tokens, {int(probe['image_grid_thw'].prod(1).sum()) // 4:,} of them "
    f"image tokens; the loss covers its last {probe['logits_to_keep'] - 1}; loss self-checks passed")

# ── The sample: 50 seeded examples per task, then the 40 longest (most images, longest answer) ──
rng = random.Random(CFG["seed"])
pools = {}
for x in examples:
    pools.setdefault(x["task"], []).append(x)
SAMPLE = [x for task in sorted(pools) for x in rng.sample(pools[task], min(CFG["per_task"], len(pools[task])))]
chosen = {id(x) for x in SAMPLE}
ranked = sorted(examples, key=lambda x: (len(x["images"]), len(x["answer"])), reverse=True)
SAMPLE += [dict(x, _longest=True) for x in [x for x in ranked if id(x) not in chosen][:CFG["longest"]]]
print(f"✓ Sample: {len(SAMPLE)} examples, {dict(Counter(x['task'] for x in SAMPLE))}")

✓ Data, option (c): 53,838 examples {'train': 53032, 'options': 806}; by task {'Disaster Bearing Bodies Recognition': 7381, 'Building Damage Counting': 14332, 'Road Damage Counting': 7227, 'disaster caption': 7381, 'Disaster Type Recognition': 1543, 'disaster restoration advice': 7380, 'Disaster Scene Recognition': 6742, 'relational reasoning': 1852}; the new 7B's checked answers in 1,079 (772 worded unlike the label)
✓ Data step in 13 s (step 2's: about 2,000 s); the images of 301 spread examples are all on disk
✓ Longest example: 1,852 tokens, 1,458 of them image tokens; the loss covers its last 355; loss self-checks passed
✓ Sample: 440 examples, {'Building Damage Counting': 50, 'Disaster Bearing Bodies Recognition': 50, 'Disaster Scene Recognition': 50, 'Disaster Type Recognition': 50, 'Road Damage Counting': 50, 'disaster caption': 90, 'disaster restoration advice': 50, 'relational reasoning': 50}


In [4]:
# ── Cell 4: Loading as step 2 does, one example's label loss, the scan, and the two fp16 fixes ──
from contextlib import nullcontext

import pandas as pd
from tqdm.auto import tqdm
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

FP16_LIMIT = float(torch.finfo(torch.float16).max) - CFG["clamp_margin"]


def load_model(compute, device):
    """Step 2's loading (4-bit NF4, double quantisation, SDPA); compute "fp16" as trained, or "fp32" for the reference."""
    dtype = torch.float16 if compute == "fp16" else torch.float32
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
                               bnb_4bit_use_double_quant=True)
    m = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE, dtype=dtype, device_map={"": device}, attn_implementation="sdpa",
                                                           quantization_config=quant)
    for mod in m.modules():          # the 4-bit layers keep the compute dtype they were given, whatever input reaches them
        if hasattr(mod, "compute_type_is_set"):
            mod.compute_type_is_set = True
    return m.eval()


def batch_of(x, device, dtype):
    """Step 2's collate for one example; the picture values in the model's dtype, as its trainer passes them."""
    b = collate([x])
    labels = b.pop("labels")
    return ({k: (v.to(device, dtype) if k == "pixel_values" else v.to(device)) if torch.is_tensor(v) else v for k, v in b.items()},
            labels.to(device))


def label_loss(model, x, device, compute):
    """(label loss on the answer tokens, seconds of forward pass, tokens, image tokens); fp16 compute runs under autocast."""
    b, labels = batch_of(x, device, torch.float16 if compute == "fp16" else torch.float32)
    ctx = torch.autocast("cuda", dtype=torch.float16) if compute == "fp16" else nullcontext()
    torch.cuda.synchronize(device)
    t = time.perf_counter()
    with torch.no_grad(), ctx:
        out = model(**b, use_cache=False)
    torch.cuda.synchronize(device)
    sec = time.perf_counter() - t
    return float(losses(out.logits, labels)[1]), sec, int(b["input_ids"].shape[1]), int(b["image_grid_thw"].prod(1).sum()) // 4


def first_tensor(out):
    if torch.is_tensor(out):
        return out
    if isinstance(out, dict):                    # a ModelOutput is a dict
        out = list(out.values())
    if isinstance(out, (tuple, list)):
        return next((o for o in out if torch.is_tensor(o)), None)
    return None


class Scan:
    """Forward hooks on every module, read once per example (nothing waits for the GPU inside the forward pass): each output's
    largest |value|, and whether its input was finite. The overflow starts at the first module, in running order, whose output
    isn't finite. If that module's input was already not finite, an operation in its parent's own code made it (an attention
    score, a residual sum), so the parent is named."""

    def __init__(self, model):
        self.names = {m: n for n, m in model.named_modules() if n}
        self.starts, self.max_abs, self.trace = Counter(), {}, []
        self.handles = [m.register_forward_hook(self._hook, with_kwargs=True) for m in self.names]

    def _hook(self, mod, args, kwargs, out):
        t = first_tensor(out)
        if t is None or not t.is_floating_point() or t.numel() == 0:
            return
        i = first_tensor(args)
        if i is None:
            i = kwargs.get("hidden_states")
        ok_in = (torch.isfinite(i).all() if torch.is_tensor(i) and i.is_floating_point() and i.numel()
                 else t.new_ones((), dtype=torch.bool))
        self.trace.append((self.names[mod], t.detach().abs().amax().float(), ok_in))

    def read(self):
        """For the example just run: (where its overflow starts or None, its largest finite output, that output's module)."""
        names = [n for n, _, _ in self.trace]
        peaks = torch.stack([v for _, v, _ in self.trace]).tolist() if self.trace else []
        ok_in = torch.stack([o for _, _, o in self.trace]).tolist() if self.trace else []
        self.trace = []
        start, peak, at = None, 0.0, ""
        for n, v, o in zip(names, peaks, ok_in):
            if math.isfinite(v):
                self.max_abs[n] = max(self.max_abs.get(n, 0.0), v)
                if v > peak:
                    peak, at = v, n
            elif start is None:
                start = n if o else n.rsplit(".", 1)[0]
        if start:
            self.starts[start] += 1
        return start, peak, at

    def remove(self):
        for h in self.handles:
            h.remove()


def fp32_islands(model, names):
    """Fix (a): these modules run in fp32 with autocast off: their inputs, their fp16 weights and their 4-bit layers' compute.
    Returns an undo that restores each exactly."""
    undo, mods = [], dict(model.named_modules())
    for n in names:
        m = mods[n]

        def forward(*args, __orig=m.forward, **kwargs):
            with torch.autocast("cuda", enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                kwargs = {k: v.float() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in kwargs.items()}
                return __orig(*args, **kwargs)

        m.forward = forward
        undo.append(lambda m=m: m.__dict__.pop("forward", None))
        for sub in m.modules():
            if hasattr(sub, "compute_dtype"):
                undo.append(lambda sub=sub, old=sub.compute_dtype: setattr(sub, "compute_dtype", old))
                sub.compute_dtype = torch.float32
            for p in sub.parameters(recurse=False):
                if p.is_floating_point() and p.dtype != torch.float32:
                    undo.append(lambda p=p, old=p.data: setattr(p, "data", old))
                    p.data = p.data.float()
    return lambda: [u() for u in reversed(undo)]


CLAMPED = re.compile(r"(language_model\.layers\.\d+|visual\.blocks\.\d+)(\.(mlp|self_attn|attn))?$")


def _clamp(out):
    if torch.is_tensor(out):
        return out.clamp(-FP16_LIMIT, FP16_LIMIT) if out.dtype == torch.float16 else out
    if isinstance(out, tuple) and out and torch.is_tensor(out[0]) and out[0].dtype == torch.float16:
        return (out[0].clamp(-FP16_LIMIT, FP16_LIMIT),) + tuple(out[1:])
    return out


def clamping(model):
    """Fix (b): fp16 outputs of every decoder layer and vision block, and of their attention and MLP, clamped (T5's fp16 guard)."""
    hs = [m.register_forward_hook(lambda mod, inp, out: _clamp(out)) for n, m in model.named_modules() if CLAMPED.search(n)]
    assert hs, "no decoder layer or vision block matched"
    return lambda: [h.remove() for h in hs]


# Self-checks on toy modules: the clamp; the scan names the parent whose own code overflowed; an fp32 island removes the
# overflow, and its undo restores the module exactly
_t = torch.tensor([1.0, 7e4, float("inf"), -float("inf")]).half()
_c = _clamp(_t)
assert torch.isfinite(_c).all() and float(_c[0]) == 1.0 and abs(float(_c[2]) - FP16_LIMIT) <= 32     # fp16 steps are 32 apart there


class _Scale(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = torch.nn.Linear(4, 4)

    def forward(self, x):
        return self.proj(x * 1e5)                # its own code overflows in fp16 before its child sees the values


_toy = torch.nn.Sequential(torch.nn.Linear(4, 4), _Scale())
torch.nn.init.ones_(_toy[0].weight)
torch.nn.init.zeros_(_toy[0].bias)
_toy = _toy.half().to("cuda:0")
_x = torch.ones(2, 4, dtype=torch.float16, device="cuda:0")
_scan = Scan(_toy)
with torch.no_grad():
    _toy(_x)
assert _scan.read()[0] == "1" and _scan.max_abs["0"] == 4.0, "the scan should name the parent whose own code overflowed"
_scan.remove()
_undo = fp32_islands(_toy, ["1"])
with torch.no_grad():
    _ok = bool(torch.isfinite(_toy(_x)).all())
_undo()
assert _ok and _toy[1].proj.weight.dtype == torch.float16 and "forward" not in _toy[1].__dict__
assert first_tensor({"a": None, "b": _t}) is _t and first_tensor((None, _t)) is _t
print(f"✓ Helpers ready and checked on toy modules; the clamp limit is {FP16_LIMIT:,.0f}")

✓ Helpers ready and checked on toy modules; the clamp limit is 64,504


In [5]:
# ── Cell 5: The scan, as trained: fp16 compute under fp16 autocast, every module watched ──
model16 = load_model("fp16", "cuda:0")
scan = Scan(model16)
rows = []
for k, x in enumerate(tqdm(SAMPLE, desc="scan, as trained (fp16)")):
    ce, sec, tokens, image_tokens = label_loss(model16, x, "cuda:0", "fp16")
    start, peak, at = scan.read()
    rows.append({"k": k, "eidx": x["eidx"], "kind": x["kind"], "task": x["task"], "longest": bool(x.get("_longest")),
                 "images": len(x["images"]), "tokens": tokens, "image_tokens": image_tokens,
                 "loss_fp16": ce, "nan_fp16": not math.isfinite(ce), "sec_fp16": round(sec, 3),
                 "overflow_starts": start or "", "peak_abs": round(peak, 1), "peak_module": at})
scan.remove()
PER = pd.DataFrame(rows)
MODS = (pd.DataFrame({"overflow_started_here": pd.Series(scan.starts, dtype=float), "max_finite_abs": pd.Series(scan.max_abs)})
        .fillna(0).sort_values(["overflow_started_here", "max_finite_abs"], ascending=False))
BAD = PER[PER["nan_fp16"]]
TOUCHED = PER[PER["overflow_starts"] != ""]
_rate = PER[~PER["longest"]].groupby("task")["nan_fp16"].mean()
WEIGHTED = float(sum(_rate[t] * by_task[t] for t in _rate.index) / sum(by_task[t] for t in _rate.index))
print(f"✓ Scan: a NaN loss in {len(BAD)} of {len(PER)} examples, a non-finite output somewhere in {len(TOUCHED)}; "
      f"weighted by task share, {WEIGHTED:.2%} of training examples (the failed run's logs imply about 1.3%)")
print(f"  Where the overflow starts: {dict(Counter(TOUCHED['overflow_starts']))}")
print(f"  NaN by task: {BAD.groupby('task').size().to_dict()}")
print(f"  Largest finite outputs: {MODS['max_finite_abs'].nlargest(5).round(0).to_dict()}")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

scan, as trained (fp16):   0%|          | 0/440 [00:00<?, ?it/s]

✓ Scan: a NaN loss in 7 of 440 examples, a non-finite output somewhere in 7; weighted by task share, 1.23% of training examples (the failed run's logs imply about 1.3%)
  Where the overflow starts: {'model.visual.blocks.31.mlp.down_proj': 7}
  NaN by task: {'Disaster Scene Recognition': 2, 'Disaster Type Recognition': 2, 'disaster restoration advice': 2, 'relational reasoning': 1}
  Largest finite outputs: {'model.visual': 64416.0, 'model.visual.blocks.31': 64416.0, 'model.visual.blocks.31.mlp.down_proj': 64352.0, 'model.visual.blocks.31.mlp': 64352.0, 'model.visual.blocks.27': 15832.0}


In [6]:
# ── Cell 6: The fixes, on every overflowing example and 15 others per task: (a) fp32 islands, (b) clamping, (c) fp32 on GPU 1 ──
_rng = random.Random(CFG["seed"] + 1)
_others = PER[(PER["overflow_starts"] == "") & ~PER["longest"]]
TEST = sorted(set(TOUCHED["k"]) | {k for _, g in _others.groupby("task")
                                   for k in _rng.sample(list(g["k"]), min(CFG["test_per_task"], len(g)))})


def run_fix(name, model, device, compute):
    for k in tqdm(TEST, desc=name):
        ce, sec, _, _ = label_loss(model, SAMPLE[k], device, compute)
        PER.loc[k, f"loss_{name}"], PER.loc[k, f"sec_{name}"] = ce, round(sec, 3)


# (a): where the overflow starts; once those run in fp32, a module further on may overflow, and it joins them
ISLANDS = sorted(set(TOUCHED["overflow_starts"]))
for _ in range(3):
    if not ISLANDS:
        break
    undo, scan = fp32_islands(model16, ISLANDS), Scan(model16)
    new = set()
    for k in TOUCHED["k"]:
        label_loss(model16, SAMPLE[k], "cuda:0", "fp16")
        new |= {scan.read()[0]} - {None}
    scan.remove()
    undo()
    if new <= set(ISLANDS):
        break
    ISLANDS = sorted(set(ISLANDS) | new)
print(f"  (a) keeps {len(ISLANDS)} module(s) in fp32: {ISLANDS}")
if ISLANDS:
    undo = fp32_islands(model16, ISLANDS)
    run_fix("a", model16, "cuda:0", "fp16")
    undo()
undo = clamping(model16)
run_fix("b", model16, "cuda:0", "fp16")
undo()
with torch.cuda.device(1):
    model32 = load_model("fp32", "cuda:1")
    run_fix("c", model32, "cuda:1", "fp32")
del model32
torch.cuda.empty_cache()

T = PER.loc[TEST]
fix_rows = []
for name, label in (("fp16", "as trained"), ("a", "(a) fp32 islands"), ("b", "(b) clamping"), ("c", "(c) fp32 compute")):
    if f"loss_{name}" not in T:
        fix_rows.append({"fix": label, "nan_losses": None, "of": len(T), "sec_per_example": None, "mean_gap_to_c": None,
                         "note": "nothing overflowed in the scan, so nothing to keep in fp32"})
        continue
    loss, ref = T[f"loss_{name}"].astype(float), T["loss_c"].astype(float)
    ok = np.isfinite(loss) & np.isfinite(ref)
    fix_rows.append({"fix": label, "nan_losses": int((~np.isfinite(loss)).sum()), "of": len(T),
                     "sec_per_example": round(float(T[f"sec_{name}"].mean()), 3),
                     "mean_gap_to_c": round(float((loss[ok] - ref[ok]).abs().mean()), 4) if ok.any() else None,
                     "note": "time includes the scan's checks" if name == "fp16" else ""})
FIXES = pd.DataFrame(fix_rows)
print(FIXES.to_string(index=False))

# The rule (fixed before the run)
_q = FIXES[FIXES["fix"].str.startswith(("(a)", "(b)")) & (FIXES["nan_losses"] == 0)
           & (FIXES["mean_gap_to_c"].fillna(1.0) <= CFG["max_loss_gap"])]
CHOSEN = _q.sort_values("sec_per_example").iloc[0]["fix"] if len(_q) else "(c) fp32 compute"
print(f"\n✓ The fix by the rule: {CHOSEN}")

  (a) keeps 1 module(s) in fp32: ['model.visual.blocks.31.mlp.down_proj']


a:   0%|          | 0/127 [00:00<?, ?it/s]

b:   0%|          | 0/127 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

c:   0%|          | 0/127 [00:00<?, ?it/s]

             fix  nan_losses  of  sec_per_example  mean_gap_to_c                            note
      as trained           7 127            2.145         0.0017 time includes the scan's checks
(a) fp32 islands           0 127            1.741         0.0017                                
    (b) clamping           0 127            1.745         0.0017                                
(c) fp32 compute           0 127            5.579         0.0000                                

✓ The fix by the rule: (a) fp32 islands


In [7]:
# ── Cell 7: Learning with the chosen fix: step 2's LoRA and the trainer's fp16 mechanics, 8 optimizer steps of 4 examples ──
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model
from transformers import get_cosine_schedule_with_warmup

del model16
torch.cuda.empty_cache()
use_fp32 = CHOSEN.startswith("(c)")
model = load_model("fp32" if use_fp32 else "fp16", "cuda:0")
PROJECTIONS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
targets = [n for n, _ in model.named_modules() if n.rsplit(".", 1)[-1] in PROJECTIONS and "visual" not in n]
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=CFG["lora"]["r"], lora_alpha=CFG["lora"]["alpha"], lora_dropout=CFG["lora"]["dropout"],
                                         target_modules=targets, bias="none", task_type="CAUSAL_LM"))
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})      # as the trainer does
if CHOSEN.startswith("(a)"):
    _full = {n.split("base_model.model.", 1)[-1]: n for n, _ in model.named_modules()}
    fp32_islands(model, [_full[n] for n in ISLANDS])
elif CHOSEN.startswith("(b)"):
    clamping(model)
model.train()
params = [p for p in model.parameters() if p.requires_grad]
assert all(p.dtype == torch.float32 for p in params), "LoRA weights must be fp32, as in step 2"
opt = bnb.optim.AdamW8bit(params, lr=CFG["lr"], betas=tuple(CFG["betas"]), weight_decay=CFG["weight_decay"])
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=1, num_training_steps=CFG["learn_steps"])
scaler = torch.amp.GradScaler("cuda", init_scale=CFG["init_scale"], enabled=not use_fp32)

# The overflowing examples first (they are what broke the run), then the other test examples
order = list(TOUCHED["k"]) + [k for k in TEST if k not in set(TOUCHED["k"])]
need = CFG["learn_steps"] * CFG["learn_accum"]
picked = (order * (need // max(len(order), 1) + 1))[:need]
steps = []
for s in range(CFG["learn_steps"]):
    parts = []
    for k in picked[s * CFG["learn_accum"]:(s + 1) * CFG["learn_accum"]]:
        b, labels = batch_of(SAMPLE[k], "cuda:0", torch.float32 if use_fp32 else torch.float16)
        with torch.autocast("cuda", dtype=torch.float16, enabled=not use_fp32):
            out = model(**b, use_cache=False)
        loss, ce, *_ = losses(out.logits, labels)
        scaler.scale(loss / CFG["learn_accum"]).backward()
        parts.append(float(ce.detach()))
    scaler.unscale_(opt)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(params, CFG["max_grad_norm"]))
    before = scaler.get_scale()
    scaler.step(opt)
    scaler.update()
    skipped = scaler.get_scale() < before          # the scaler halves its scale exactly when it skips the update
    if not skipped:
        sched.step()                               # as the trainer: the learning rate moves only after a real update
    opt.zero_grad(set_to_none=True)
    steps.append({"step": s + 1, "label_loss": round(float(np.mean(parts)), 4), "grad_norm": grad_norm,
                  "learning_rate": sched.get_last_lr()[0], "scale": scaler.get_scale(), "skipped": skipped})
    print(steps[-1], flush=True)
STEPS = pd.DataFrame(steps)
LEARNS = bool((STEPS["learning_rate"] > 0).any() and np.isfinite(STEPS["grad_norm"].tail(3)).all())
print(f"\n{'✓' if LEARNS else '✗'} Learning with {CHOSEN}: a real update, and a finite grad_norm in the last 3 steps: {LEARNS}")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

{'step': 1, 'label_loss': 1.5091, 'grad_norm': 0.898189127445221, 'learning_rate': 0.0002, 'scale': 65536.0, 'skipped': False}
{'step': 2, 'label_loss': 1.7928, 'grad_norm': 0.34062397480010986, 'learning_rate': 0.0001900968867902419, 'scale': 65536.0, 'skipped': False}
{'step': 3, 'label_loss': 0.9329, 'grad_norm': 0.3594242036342621, 'learning_rate': 0.00016234898018587337, 'scale': 65536.0, 'skipped': False}
{'step': 4, 'label_loss': 1.0401, 'grad_norm': 0.43454858660697937, 'learning_rate': 0.00012225209339563145, 'scale': 65536.0, 'skipped': False}
{'step': 5, 'label_loss': 0.7946, 'grad_norm': 0.38334155082702637, 'learning_rate': 7.774790660436858e-05, 'scale': 65536.0, 'skipped': False}
{'step': 6, 'label_loss': 0.9382, 'grad_norm': 0.3573300540447235, 'learning_rate': 3.7651019814126654e-05, 'scale': 65536.0, 'skipped': False}
{'step': 7, 'label_loss': 1.3113, 'grad_norm': 0.4481796324253082, 'learning_rate': 9.903113209758096e-06, 'scale': 65536.0, 'skipped': False}
{'step': 

In [8]:
# ── Cell 8: The files and the README ──
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


PER.to_csv(WORK / "per_example.csv", index=False)
MODS.rename_axis("module").reset_index().to_csv(WORK / "modules.csv", index=False)
FIXES.to_csv(WORK / "fixes.csv", index=False)
STEPS.to_csv(WORK / "train_steps.csv", index=False)
sec = float(FIXES.loc[FIXES["fix"] == CHOSEN, "sec_per_example"].iloc[0])
epoch_h = sec * len(examples) / 2 / 3600
cfg = dict(CFG, chosen_fix=CHOSEN, learns=LEARNS, scanned=len(PER), nan_as_trained=len(BAD), overflowing=len(TOUCHED),
           weighted_nan_share=round(WEIGHTED, 4), islands=ISLANDS, test_examples=len(TEST), data_step_seconds=round(DATA_SECONDS),
           forward_only_epoch_hours=round(epoch_h, 1), versions=VERSIONS)
(WORK / "config.json").write_text(json.dumps(cfg, indent=1, default=str), encoding="utf-8")
top = MODS.sort_values("max_finite_abs", ascending=False).head(10).rename_axis("module").reset_index()
(WORK / "README.md").write_text("\n".join([
    f"# L2 step 2b: fp16 overflow in the 3B's training{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    "The question, the rule and the expectation, fixed before the run, are in the notebook's first cell.", "",
    "## The scan, as trained (fp16 compute under fp16 autocast)", "",
    f"{len(PER)} option (c) training examples: {CFG['per_task']} seeded per task and the {CFG['longest']} longest.", "",
    f"- A NaN label loss in **{len(BAD)}** of them; a non-finite output somewhere in {len(TOUCHED)}.",
    f"- Weighted by each task's share of the {len(examples):,} training examples: **{WEIGHTED:.2%}**. The failed run's logs "
    "imply about 1.3%: 13 of its 16 steps had a NaN among the first GPU's 128 examples.",
    f"- NaN by task: {BAD.groupby('task').size().to_dict()}.",
    f"- Where the overflow starts (examples): {dict(Counter(TOUCHED['overflow_starts']))}.", "",
    "The largest finite outputs (10 modules):", "", md(top), "",
    f"## The fixes, on {len(TEST)} examples (every overflowing one and {CFG['test_per_task']} others per task)", "",
    f"(a) kept {len(ISLANDS)} module(s) in fp32: {ISLANDS}.", "", md(FIXES), "",
    f"**The fix by the rule:** {CHOSEN}.", "",
    f"## Learning with it ({CFG['learn_steps']} optimizer steps of {CFG['learn_accum']} examples, the trainer's fp16 mechanics)", "",
    md(STEPS), "", f"**Learning works:** {LEARNS}.", "",
    "## Time", "",
    f"- With {CHOSEN}, the forward pass alone takes {sec:.2f} s per example on one T4. An epoch of {len(examples):,} on two T4s "
    f"then needs at least {epoch_h:.1f} h before the backward pass. Step 2's run took 912 s per step of 256: 7.1 s per example "
    "on each T4, backward pass included (53 h per epoch).",
    f"- The data step took {DATA_SECONDS:.0f} s with folder listings (step 2's: about 2,000 s).", "",
    "Files: `per_example.csv`, `modules.csv`, `fixes.csv`, `train_steps.csv`, `config.json`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, config.json, per_example.csv, modules.csv, fixes.csv and train_steps.csv to {WORK}")

✓ Wrote README.md, config.json, per_example.csv, modules.csv, fixes.csv and train_steps.csv to /kaggle/working/L2S2b
